# PIPELINE TIỀN XỬ LÝ DỮ LIỆU DỰ BÁO PHỤ TẢI ĐIỆN 30 PHÚT
### (Cấu trúc đầu ra chuẩn mực: Tách bạch Data, Reports và Figures)

**Mục tiêu cấu trúc lưu trữ:**
- `output_preprocessing/data/`: Chứa toàn bộ các tệp dữ liệu đã làm sạch (`cleaned_long.csv`, `cleaned_wide.csv`, `train.csv`, `validation.csv`, `test.csv`).
- `output_preprocessing/reports/`: Chứa các bảng báo cáo định lượng (`quality_report.csv`, `station_summary.csv`, `preprocessing_summary.json`).
- `output_preprocessing/figures/`: Chứa **8 biểu đồ đánh giá chất lượng cao (200 DPI)** phục vụ báo cáo khoa học và luận văn.

## 1. Thiết lập thư mục đầu vào và khởi tạo các thư mục đầu ra riêng biệt

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình font và thẩm mỹ cho biểu đồ đánh giá
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 1. Nhận diện đường dẫn dữ liệu đầu vào
if Path("/content/drive/MyDrive/Tam_TLU/datasets").exists():
    DATA_DIR = Path("/content/drive/MyDrive/Tam_TLU/datasets")
elif Path("/content/drive/MyDrive/power-load-forecasting/Tam_TLU/datasets").exists():
    DATA_DIR = Path("/content/drive/MyDrive/power-load-forecasting/Tam_TLU/datasets")
elif Path("../datasets").exists():
    DATA_DIR = Path("../datasets")
else:
    DATA_DIR = Path("./datasets")

# 2. Khởi tạo 3 thư mục đầu ra riêng biệt
BASE_OUTPUT = Path("./output_preprocessing")
DATA_OUT = BASE_OUTPUT / "data"
REPORTS_OUT = BASE_OUTPUT / "reports"
FIGURES_OUT = BASE_OUTPUT / "figures"

for d in [DATA_OUT, REPORTS_OUT, FIGURES_OUT]:
    d.mkdir(parents=True, exist_ok=True)

NUMERIC_COLS = ["UA", "UB", "UC", "IA", "IB", "IC", "CosF", "P_kW", "Q_kVAr"]

print(f"Thư mục dữ liệu gốc:      {DATA_DIR.resolve()}")
print(f"Thư mục dữ liệu sạch:     {DATA_OUT.resolve()}")
print(f"Thư mục báo cáo CSV:      {REPORTS_OUT.resolve()}")
print(f"Thư mục biểu đồ đánh giá: {FIGURES_OUT.resolve()}")

## 2. Quét toàn bộ file CSV đo đếm (Bỏ qua file nén zip/7z)

In [ ]:
all_csv_paths = sorted(DATA_DIR.rglob("*.csv"))
exclude_kw = ["report", "summary", "cleaned", "featured", "supervised", "train", "validation", "test"]
valid_csvs = [p for p in all_csv_paths if not any(kw in p.name.lower() for kw in exclude_kw)]

print(f"Đã tìm thấy {len(valid_csvs)} file CSV đo đếm hợp lệ.")
for p in valid_csvs[:3]:
    print(f"  - Mẫu: {p.name}")

## 3. Đọc, chuẩn hóa tên cột, kiểu dữ liệu thời gian và số liệu

In [ ]:
def parse_time_robust(df):
    time_col = next((c for c in ["TIME", "Time", "ThoiGian", "THOIGIAN"] if c in df.columns), None)
    if not time_col:
        raise ValueError(f"Thiếu cột thời gian trong: {df.columns.tolist()}")
    s = df[time_col].astype(str).str.strip()
    parsed = pd.to_datetime(s, errors="coerce")
    if parsed.notna().mean() >= 0.8:
        return parsed
    date_col = next((c for c in ["DATE", "Date", "Ngay", "NGAY"] if c in df.columns), None)
    if date_col:
        s_date = pd.to_datetime(df[date_col], errors="coerce")
        s_fixed = s.str.replace(r"\s+", "", regex=True)
        s_fixed = s_fixed.where(s_fixed.str.count(":") == 2, "00:" + s_fixed)
        return s_date + pd.to_timedelta(s_fixed, errors="coerce")
    return parsed

dfs = []
for p in valid_csvs:
    try:
        df_i = pd.read_csv(p, low_memory=False, encoding="utf-8")
    except UnicodeDecodeError:
        df_i = pd.read_csv(p, low_memory=False, encoding="latin1")
    
    df_i.columns = [str(c).strip() for c in df_i.columns]
    df_i["TIME"] = parse_time_robust(df_i)
    df_i = df_i.dropna(subset=["TIME"]).copy()
    
    for c in NUMERIC_COLS:
        if c in df_i.columns:
            s_num = df_i[c].astype(str).str.strip().str.replace(",", ".", regex=False)
            df_i[c] = pd.to_numeric(s_num, errors="coerce")
            
    if "MaDiemDo" in df_i.columns:
        df_i["MaDiemDo"] = pd.to_numeric(df_i["MaDiemDo"], errors="coerce")
        df_i = df_i.dropna(subset=["MaDiemDo"]).copy()
        df_i["MaDiemDo"] = df_i["MaDiemDo"].astype(int)
    if "TenDiemDo" in df_i.columns:
        df_i["TenDiemDo"] = df_i["TenDiemDo"].astype(str).str.strip()
        
    df_i["source_file"] = p.name
    dfs.append(df_i)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Tổng số dòng dữ liệu thô đã tải: {len(df_raw):,}")

## 4. Kiểm tra chất lượng và xuất `reports/quality_report.csv`

In [ ]:
total_rows = len(df_raw)
missing_stats = df_raw.isna().sum()
missing_pct = (missing_stats / total_rows * 100).round(2)

report_rows = []
for c in df_raw.columns:
    report_rows.append({
        "column_name": c,
        "data_type": str(df_raw[c].dtype),
        "missing_count": int(missing_stats[c]),
        "missing_pct": float(missing_pct[c]),
        "unique_values": int(df_raw[c].nunique())
    })

df_quality_report = pd.DataFrame(report_rows)
df_quality_report.to_csv(REPORTS_OUT / "quality_report.csv", index=False, encoding="utf-8-sig")
print(f"Đã xuất bảng đánh giá chất lượng tại: {REPORTS_OUT / 'quality_report.csv'}")

## 5. Quy chuẩn lưới 30 phút, nội suy có kiểm soát và lưu vào `data/`

In [ ]:
df_raw["TIME_30m"] = df_raw["TIME"].dt.round("30min")
df_sorted = df_raw.sort_values(by=["MaDiemDo", "TIME_30m", "TIME"]).reset_index(drop=True)
df_dedup = df_sorted.drop_duplicates(subset=["MaDiemDo", "TIME_30m"], keep="first").copy()

resampled_list = []
for point_id, group in df_dedup.groupby("MaDiemDo"):
    group = group.set_index("TIME_30m").sort_index()
    timeline = pd.date_range(start=group.index.min(), end=group.index.max(), freq="30min", name="TIME")
    
    reindexed = group.reindex(timeline)
    reindexed["MaDiemDo"] = point_id
    if "TenDiemDo" in group.columns:
        reindexed["TenDiemDo"] = group["TenDiemDo"].dropna().iloc[0] if len(group["TenDiemDo"].dropna()) > 0 else f"Point_{point_id}"
    if "MBA" in group.columns:
        reindexed["MBA"] = group["MBA"].dropna().iloc[0] if len(group["MBA"].dropna()) > 0 else "Unknown"
        
    was_missing = reindexed["P_kW"].isna()
    present_nums = [c for c in NUMERIC_COLS if c in reindexed.columns]
    
    # Chỉ nội suy tối đa 2 bước liên tiếp (1 tiếng)
    imputed = reindexed[present_nums].interpolate(method="time", limit=2, limit_direction="forward")
    reindexed["is_imputed"] = (was_missing & imputed["P_kW"].notna()).astype(int)
    reindexed[present_nums] = imputed
    
    resampled_list.append(reindexed.reset_index().rename(columns={"index": "TIME"}))

df_clean_long = pd.concat(resampled_list, ignore_index=True)
df_clean_long.to_csv(DATA_OUT / "cleaned_long.csv", index=False, encoding="utf-8-sig")

# Xuất cleaned_wide.csv
df_wide = df_clean_long.pivot_table(index="TIME", columns="MaDiemDo", values=["P_kW", "Q_kVAr"], aggfunc="first")
df_wide.columns = [f"{var}_{pt}" for (var, pt) in df_wide.columns]
df_wide.reset_index().to_csv(DATA_OUT / "cleaned_wide.csv", index=False, encoding="utf-8-sig")

print(f"Đã lưu: {DATA_OUT / 'cleaned_long.csv'}")
print(f"Đã lưu: {DATA_OUT / 'cleaned_wide.csv'}")

## 6. Trích xuất đặc trưng & Tổng hợp danh sách trạm vào `reports/`

In [ ]:
df_feat = df_clean_long.copy()
eps = 1e-6

# Đặc trưng 3 pha
if set(["UA", "UB", "UC"]).issubset(df_feat.columns):
    df_feat["U_avg"] = df_feat[["UA", "UB", "UC"]].mean(axis=1)
    df_feat["U_max"] = df_feat[["UA", "UB", "UC"]].max(axis=1)
    df_feat["U_min"] = df_feat[["UA", "UB", "UC"]].min(axis=1)
    df_feat["U_unbalance_pct"] = (df_feat["U_max"] - df_feat["U_min"]) / (df_feat["U_avg"].abs() + eps) * 100

if set(["IA", "IB", "IC"]).issubset(df_feat.columns):
    df_feat["I_avg"] = df_feat[["IA", "IB", "IC"]].mean(axis=1)
    df_feat["I_max"] = df_feat[["IA", "IB", "IC"]].max(axis=1)
    df_feat["I_min"] = df_feat[["IA", "IB", "IC"]].min(axis=1)
    df_feat["I_unbalance_pct"] = (df_feat["I_max"] - df_feat["I_min"]) / (df_feat["I_avg"].abs() + eps) * 100

df_feat["S_kVA"] = np.sqrt(df_feat["P_kW"].fillna(0)**2 + df_feat["Q_kVAr"].fillna(0)**2)
df_feat["hour"] = df_feat["TIME"].dt.hour + df_feat["TIME"].dt.minute / 60.0
df_feat["dayofweek"] = df_feat["TIME"].dt.dayofweek
df_feat["dayofmonth"] = df_feat["TIME"].dt.day
df_feat["month"] = df_feat["TIME"].dt.month
df_feat["is_weekend"] = (df_feat["dayofweek"] >= 5).astype(int)
df_feat["sin_hour"] = np.sin(2 * np.pi * df_feat["hour"] / 24.0)
df_feat["cos_hour"] = np.cos(2 * np.pi * df_feat["hour"] / 24.0)

feat_groups = []
for point_id, grp in df_feat.groupby("MaDiemDo"):
    grp = grp.sort_values("TIME").copy()
    for lag in [1, 2, 3, 48, 336]:
        grp[f"lag_{lag}"] = grp["P_kW"].shift(lag)
    p_prev = grp["P_kW"].shift(1)
    grp["rolling_mean_3"] = p_prev.rolling(3, min_periods=1).mean()
    grp["rolling_mean_6"] = p_prev.rolling(6, min_periods=1).mean()
    grp["rolling_mean_48"] = p_prev.rolling(48, min_periods=1).mean()
    feat_groups.append(grp)

df_featured = pd.concat(feat_groups, ignore_index=True)
df_featured.to_csv(DATA_OUT / "featured_data.csv", index=False, encoding="utf-8-sig")

# Tổng hợp trạm
station_rows = []
for pid, grp in df_clean_long.groupby("MaDiemDo"):
    name = grp["TenDiemDo"].iloc[0] if "TenDiemDo" in grp.columns else f"Point_{pid}"
    station_rows.append({
        "MaDiemDo": pid,
        "TenDiemDo": name,
        "total_records": len(grp),
        "P_mean_kW": round(float(grp["P_kW"].mean()), 3),
        "P_max_kW": round(float(grp["P_kW"].max()), 3),
        "missing_rate_pct": round(float(grp["P_kW"].isna().mean() * 100), 2)
    })

df_station_summary = pd.DataFrame(station_rows).sort_values(by="P_mean_kW", ascending=False).reset_index(drop=True)
df_station_summary.to_csv(REPORTS_OUT / "station_summary.csv", index=False, encoding="utf-8-sig")
TARGET_POINT = int(df_station_summary.iloc[0]["MaDiemDo"])
print(f"Trạm mục tiêu lớn nhất được chọn: MaDiemDo = {TARGET_POINT} ({df_station_summary.iloc[0]['TenDiemDo']})")

## 7. Thiết lập bài toán học có giám sát và chia tập dữ liệu vào `data/`

In [ ]:
df_target = df_featured[df_featured["MaDiemDo"] == TARGET_POINT].sort_values("TIME").copy()
df_target["target_P_kW"] = df_target["P_kW"].shift(-1)
df_supervised = df_target.dropna(subset=["target_P_kW"]).copy()

df_supervised["target_delta_P"] = df_supervised["target_P_kW"] - df_supervised["P_kW"]
threshold_kw = 0.5
conditions = [
    df_supervised["target_delta_P"] > threshold_kw,
    df_supervised["target_delta_P"] < -threshold_kw
]
df_supervised["target_direction"] = np.select(conditions, ["TĂNG", "GIẢM"], default="KHÔNG ĐỔI")

n = len(df_supervised)
t_idx = int(n * 0.70)
v_idx = int(n * 0.85)

train_df = df_supervised.iloc[:t_idx].copy()
val_df = df_supervised.iloc[t_idx:v_idx].copy()
test_df = df_supervised.iloc[v_idx:].copy()

df_supervised.to_csv(DATA_OUT / "supervised_data.csv", index=False, encoding="utf-8-sig")
train_df.to_csv(DATA_OUT / "train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(DATA_OUT / "validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(DATA_OUT / "test.csv", index=False, encoding="utf-8-sig")

print(f"Đã lưu các tập dữ liệu tại: {DATA_OUT.resolve()}")
print(f"- Train: {len(train_df):,} dòng | Val: {len(val_df):,} dòng | Test: {len(test_df):,} dòng")

## 8. Tự động sinh BỘ 8 BIỂU ĐỒ ĐÁNH GIÁ CHẤT LƯỢNG CAO vào `figures/`

In [ ]:
clean_target = df_clean_long[df_clean_long["MaDiemDo"] == TARGET_POINT].set_index("TIME").sort_index()
raw_target = df_raw[df_raw["MaDiemDo"] == TARGET_POINT].set_index("TIME").sort_index()

# --- 1. Toàn cảnh chuỗi công suất & Phục hồi nội suy ---
fig1, (ax1a, ax1b) = plt.subplots(2, 1, figsize=(16, 9))
sample_month = clean_target.iloc[:1440]
ax1a.plot(sample_month.index, sample_month["P_kW"], color="#1f77b4", lw=1.2, label=f"Trạm {TARGET_POINT} (30 ngày)")
ax1a.set_title("1A. Toàn cảnh chuỗi phụ tải P (kW) sau làm sạch (Khung 30 ngày)", fontsize=13, fontweight="bold")
ax1a.legend(loc="upper right")

t0 = clean_target.index.min()
t1 = t0 + pd.Timedelta(days=7)
ax1b.plot(raw_target.loc[t0:t1].index, raw_target.loc[t0:t1, "P_kW"], "r.", markersize=8, alpha=0.6, label="Gốc (Thiếu/Lệch giây)")
ax1b.plot(clean_target.loc[t0:t1].index, clean_target.loc[t0:t1, "P_kW"], color="#2ca02c", lw=1.5, alpha=0.9, label="Đã xử lý (Lưới 30m, Nội suy <=2 bước)")
ax1b.set_title("1B. Kiểm chứng phục hồi dữ liệu: So sánh Gốc vs Sau xử lý (Zoom 7 ngày)", fontsize=13, fontweight="bold")
ax1b.legend(loc="upper right")
fig1.tight_layout()
fig1.savefig(FIGURES_OUT / "01_load_timeline_and_imputation.png", dpi=200)
plt.show()

# --- 2. Tỷ lệ thiếu dữ liệu ---
fig2, ax2 = plt.subplots(figsize=(11, 4.5))
missing_p = (df_raw[NUMERIC_COLS].isna().mean() * 100).sort_values(ascending=True)
bars2 = ax2.barh(missing_p.index, missing_p.values, color=sns.color_palette("Blues_r", len(missing_p)))
ax2.set_title("2. Tỷ lệ khuyết dữ liệu gốc theo từng biến số (%)", fontsize=13, fontweight="bold")
for b in bars2:
    w = b.get_width()
    ax2.text(w + 0.3, b.get_y() + b.get_height()/2, f"{w:.2f}%", va="center", fontsize=10, fontweight="bold")
fig2.tight_layout()
fig2.savefig(FIGURES_OUT / "02_data_completeness_missing_matrix.png", dpi=200)
plt.show()

# --- 3. Phân phối & Boxplot ---
fig3, axes3 = plt.subplots(1, 2, figsize=(15, 5.5))
p_data = clean_target["P_kW"].dropna()
sns.histplot(p_data, bins=60, kde=True, ax=axes3[0], color="#1f77b4", edgecolor="black", alpha=0.6)
axes3[0].axvline(p_data.mean(), color="red", linestyle="--", lw=2, label=f"Mean = {p_data.mean():.2f}")
axes3[0].axvline(p_data.median(), color="green", linestyle="-.", lw=2, label=f"Median = {p_data.median():.2f}")
axes3[0].set_title("Phân phối xác suất (Histogram & KDE) của P_kW", fontsize=12, fontweight="bold")
axes3[0].legend()
sns.boxplot(y=p_data, ax=axes3[1], color="#98df8a")
axes3[1].set_title("Biểu đồ Boxplot: Khoảng tứ phân vị & Ngoại lai", fontsize=12, fontweight="bold")
fig3.tight_layout()
fig3.savefig(FIGURES_OUT / "03_power_distribution_and_anomaly.png", dpi=200)
plt.show()

# --- 4. Ma trận tương quan đặc trưng ---
fig4, ax4 = plt.subplots(figsize=(10, 7))
corr_cols = [c for c in ["P_kW", "target_P_kW", "lag_1", "lag_2", "lag_48", "rolling_mean_3", "U_avg", "I_avg", "S_kVA", "CosF"] if c in df_supervised.columns]
sns.heatmap(df_supervised[corr_cols].corr(), annot=True, fmt=".2f", cmap="vlag", ax=ax4)
ax4.set_title("4. Ma trận tương quan đặc trưng với biến mục tiêu (target_P_kW)", fontsize=13, fontweight="bold")
fig4.tight_layout()
fig4.savefig(FIGURES_OUT / "04_correlation_heatmap_features.png", dpi=200)
plt.show()

# --- 5. Đánh giá biến mục tiêu (Delta P & Direction) ---
fig5, axes5 = plt.subplots(1, 2, figsize=(15, 5.5))
sns.histplot(df_supervised["target_delta_P"].dropna(), bins=70, kde=True, ax=axes5[0], color="#9467bd")
axes5[0].axvline(0, color="red", linestyle="--", lw=1.5)
axes5[0].set_title("Phân phối Delta P = P[t+1] - P[t] (kW)", fontsize=12, fontweight="bold")

d_counts = df_supervised["target_direction"].value_counts()
bars5 = axes5[1].bar(d_counts.index, d_counts.values, color=["#2ca02c", "#d62728", "#7f7f7f"][:len(d_counts)], edgecolor="black")
axes5[1].set_title("Tỷ lệ nhãn xu hướng (TĂNG / GIẢM / KHÔNG ĐỔI)", fontsize=12, fontweight="bold")
for b in bars5:
    h = b.get_height()
    axes5[1].text(b.get_x() + b.get_width()/2, h + 10, f"{h:,}\n({h/len(df_supervised)*100:.1f}%)", ha="center", fontsize=10, fontweight="bold")
fig5.tight_layout()
fig5.savefig(FIGURES_OUT / "05_target_evaluation_and_direction.png", dpi=200)
plt.show()

# --- 6. Scatter P[t] vs P[t+1] ---
fig6, ax6 = plt.subplots(figsize=(7, 7))
samp = df_supervised.sample(n=min(3000, len(df_supervised)), random_state=42)
ax6.scatter(samp["P_kW"], samp["target_P_kW"], alpha=0.3, color="#1f77b4", s=25)
m_val = min(samp["P_kW"].min(), samp["target_P_kW"].min())
mx_val = max(samp["P_kW"].max(), samp["target_P_kW"].max())
ax6.plot([m_val, mx_val], [m_val, mx_val], "r--", lw=2, label="Đường y = x")
r = df_supervised[["P_kW", "target_P_kW"]].corr().iloc[0, 1]
ax6.set_title(f"6. Tương quan tuyến tính P[t] vs P[t+1] (r = {r:.4f})", fontsize=13, fontweight="bold")
ax6.legend(loc="upper left")
fig6.tight_layout()
fig6.savefig(FIGURES_OUT / "06_scatter_lag_vs_target.png", dpi=200)
plt.show()

# --- 7. Tính mùa vụ Giờ & Thứ ---
fig7, axes7 = plt.subplots(1, 2, figsize=(16, 5.5))
h_m = clean_target.groupby(clean_target.index.hour)["P_kW"].mean()
h_s = clean_target.groupby(clean_target.index.hour)["P_kW"].std()
axes7[0].plot(h_m.index, h_m.values, marker="o", color="#ff7f0e", lw=2.5)
axes7[0].fill_between(h_m.index, h_m.values - h_s.values, h_m.values + h_s.values, color="#ff7f0e", alpha=0.2)
axes7[0].set_title("Phụ tải trung bình theo 24 giờ trong ngày", fontsize=12, fontweight="bold")
axes7[0].set_xticks(range(0, 24))

d_m = clean_target.groupby(clean_target.index.dayofweek)["P_kW"].mean()
axes7[1].bar(["T2", "T3", "T4", "T5", "T6", "T7", "CN"], d_m.values, color=sns.color_palette("Set2", 7), edgecolor="black")
axes7[1].set_title("Phụ tải trung bình theo ngày trong tuần", fontsize=12, fontweight="bold")
fig7.tight_layout()
fig7.savefig(FIGURES_OUT / "07_daily_and_weekly_seasonality.png", dpi=200)
plt.show()

# --- 8. Timeline chia tập Train - Val - Test ---
fig8, ax8 = plt.subplots(figsize=(15, 3.8))
df_s = df_supervised.sort_values("TIME").reset_index(drop=True)
ax8.plot(df_s.iloc[:t_idx]["TIME"], df_s.iloc[:t_idx]["P_kW"], color="#1f77b4", label="Train (70%)")
ax8.plot(df_s.iloc[t_idx:v_idx]["TIME"], df_s.iloc[t_idx:v_idx]["P_kW"], color="#ff7f0e", label="Validation (15%)")
ax8.plot(df_s.iloc[v_idx:]["TIME"], df_s.iloc[v_idx:]["P_kW"], color="#2ca02c", label="Test (15%)")
ax8.set_title("8. Phân chia tập dữ liệu tuần tự theo thời gian (No Leakage)", fontsize=13, fontweight="bold")
ax8.legend(loc="upper right")
fig8.tight_layout()
fig8.savefig(FIGURES_OUT / "08_data_split_timeline.png", dpi=200)
plt.show()

print(f"ĐÃ LƯU TOÀN BỘ 8 BIỂU ĐỒ ĐÁNH GIÁ (DPI 200) TẠI: {FIGURES_OUT.resolve()}")